# Projet de groupe Web scraping : véhicules d'occasion (Ottawa-Gatineau)

**Cours :** Collecte et nettoyage des données (projet UA2)
**Groupe :** Yacine Lezoul (2748526) et Mouncif Bouchachi (2743259)
**Activité :** concessionnaires de véhicules d'occasion de la région Ottawa-Gatineau

Ce notebook (le premier des deux) suit la démarche du document « Projet de groupe Web scraping ». Le code est rangé
dans des modules de `src/` ; le notebook les appelle et affiche les résultats avec les explications.

| Partie du cours | Contenu | Section | Module |
|---|---|---|---|
| 0 | Préparation (environnement, User-Agent, délais) | 0 | `config.py` |
| 1 | Cadre légal et éthique (5 questions, robots.txt, conditions, API) | 1 | `config.py`, `diagnostic.py` |
| 2 | Diagnostic de chaque page (cas A à I) | 2 | `diagnostic.py` |
| 3 et 4 | Collecte et pipeline | 3 | `extraction.py`, `pipeline.py` |
| 5 | Nettoyage, fusion, tables finales, rapport de qualité | notebook `02_pretraitement.ipynb` | `pretraitement.py` |
| 5 | Paragraphe par URL, vérification finale | 4 et 5 | |

**Organisation des dossiers**

```
Projet_UA2_ETL/
├── notebook/
│   ├── 01_collecte.ipynb        # ce notebook : parties 0 à 3 (préparation, éthique, diagnostic, collecte)
│   └── 02_pretraitement.ipynb   # nettoyage, complément par le NIV, fusion des collectes, tables pour Power BI
├── src/
│   ├── config.py                # réglages : chemins, User-Agent, délais, sites, cas confirmés
│   ├── diagnostic.py            # parties 1 et 2 : URL sans doublons, diagnostic, API, journal
│   ├── extraction.py            # partie 3 : fonction unique d'extraction, pagination, CSV par site
│   ├── enrichissement.py        # décodage des nouveaux NIV par l'API officielle vPIC (NHTSA)
│   ├── pretraitement.py         # nettoyage et fusion : observations + véhicules consolidés
│   ├── pipeline.py              # tout le projet en une commande : python src/pipeline.py
│   ├── projet_scraping.py       # boîte à outils fournie par le cours
│   ├── scraper.py               # pipeline fourni par le cours (référence)
│   └── spider_livres.py         # exemple Scrapy fourni par le cours (référence)
├── pages_brutes/                # HTML téléchargé au diagnostic (non publié sur GitHub)
├── resultats/
│   ├── brut/                    # un CSV par site et par date de collecte
│   ├── vehicules_brut_AAAA-MM-JJ.csv   # tous les véhicules d'une collecte
│   ├── historique_collectes.csv        # une ligne par site et par collecte (jamais écrasé)
│   ├── journal_tests.csv               # journal de tests (annexe C)
│   ├── niv_decodes_vpic.csv            # cache des NIV décodés par l'API vPIC
│   ├── observations_quotidiennes.csv   # prétraitement : 1 véhicule, 1 site, 1 jour
│   ├── vehicules_occasion_consolide.csv # prétraitement : 1 véhicule (NIV)
│   ├── concessionnaires.csv            # prétraitement : 1 site (dimension Power BI)
│   └── rapport_qualite.csv             # prétraitement : contrôles de qualité
├── scripts/                     # collecte_quotidienne.bat + installer_tache.ps1 (chaque nuit à 3 h)
├── logs/                        # un log par exécution automatique (non publié)
├── requirements.txt             # versions des bibliothèques
└── .gitignore
```

**Exécution.** Dans VS Code, choisir le noyau « Python (projet-scraping) », puis **Run All**
(bouton « Exécuter tout » en haut du notebook). Le même travail peut être lancé sans notebook :
`python src/pipeline.py` depuis la racine du projet.

## 0. Préparation (partie 0 du cours)

### 0.1 Chemins et importation des modules

La cellule retrouve la racine du projet (le dossier qui contient `src/`), quel que soit le dossier
depuis lequel le notebook est lancé, puis importe la boîte à outils du cours et nos modules.

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd


def trouver_racine(depart=None):
    """Remonte les dossiers jusqu'à trouver celui qui contient src/projet_scraping.py."""
    dossier = Path(depart or Path.cwd()).resolve()
    for candidat in [dossier, *dossier.parents]:
        if (candidat / "src" / "projet_scraping.py").exists():
            return candidat
    raise FileNotFoundError("Dossier src/ introuvable : lancez le notebook depuis le projet.")


RACINE = trouver_racine()
if str(RACINE / "src") not in sys.path:
    sys.path.insert(0, str(RACINE / "src"))

import projet_scraping as ws      # boîte à outils du cours
import config                     # nos réglages
import diagnostic                 # parties 1 et 2
import extraction                 # partie 3

pd.set_option("display.max_colwidth", 120)
print("Racine du projet :", config.RACINE)

### 0.2 Version de Python

Le document « Avant de scraper » (section 3.1) précise que `urllib.robotparser` applique
correctement robots.txt seulement à partir de **Python 3.13.14** ou **3.14.5**. Avec une version
plus ancienne, une page interdite peut être déclarée permise : on vérifie avant toute collecte.

In [ ]:
v = sys.version_info
version_ok = (v[:2] == (3, 13) and v >= (3, 13, 14)) or (v[:2] == (3, 14) and v >= (3, 14, 5)) or v[:2] > (3, 14)
print(f"Python {v.major}.{v.minor}.{v.micro} :", "conforme" if version_ok else "TROP ANCIENNE, mettre Python à jour")
assert version_ok, "Mettre Python à jour avant de continuer (voir « Avant de scraper », section 3.1)."

### 0.3 Réglages utilisés partout (`src/config.py`)

| Réglage | Valeur | Pourquoi |
|---|---|---|
| `USER_AGENT` | nom du projet + nos deux courriels | Nous identifier honnêtement : un administrateur peut nous contacter au lieu de nous bloquer |
| `DELAI_MINIMUM` | 2 s | Le cours exige au moins 1 s entre deux requêtes au même site |
| Crawl-delay | lu dans robots.txt (5 s pour les sites D2C Media) | On applique toujours le plus grand des délais |
| `MAX_PAGES_PAR_SITE` | 20 | Garde-fou : jamais plus de 20 pages de liste par site |
| `TIMEOUT` | 15 s (valeur de `projet_scraping.py`) | Abandonner une requête qui ne répond pas |

In [ ]:
session = ws.creer_session(config.USER_AGENT)
print("User-Agent :", session.headers["User-Agent"])
print("Délai minimum :", config.DELAI_MINIMUM, "s ; pages max par site :", config.MAX_PAGES_PAR_SITE,
      "; timeout :", ws.TIMEOUT, "s")

## 1. Cadre légal et éthique (partie 1 du cours)

### 1.1 Les cinq questions avant toute collecte

| Question du cours | Notre réponse |
|---|---|
| 1. Existe-t-il une API officielle ou un jeu de données ouvert ? | Non. Aucun lien « API » ou « données » sur les sites, aucun jeu de données d'inventaire de concessionnaires sur ouvert.canada.ca ni sur donneesquebec.ca (Statistique Canada publie seulement des ventes agrégées). Il existe des fournisseurs commerciaux payants (MarketCheck, VinAudit), ni officiels ni ouverts. Vérification automatique complémentaire à la section 2. **Pour le décodage du NIV, une API officielle existe** : vPIC de la NHTSA (département des Transports des États-Unis), gratuite et ouverte au public. Elle est utilisée pour compléter le carburant, la catégorie et la traction (module `enrichissement.py`, notebook `02_pretraitement.ipynb`). |
| 2. Que dit robots.txt ? | Lu pour chaque site (tableau 1.2), puis revérifié automatiquement par `robots_autorise()` avant chaque requête. |
| 3. Que disent les conditions d'utilisation ? | Lues pour chaque site. Les sites qui interdisent les robots ou le moissonnage sont exclus (section 1.4). |
| 4. Y a-t-il des renseignements personnels ? | Non : un inventaire décrit des produits (prix, année, kilométrage). On ne collecte ni noms ni téléphones. |
| 5. Mon script est-il poli ? | User-Agent identifiable, délai d'au moins 2 s (5 s si robots.txt le demande), pages de liste seulement, arrêt immédiat en cas de blocage. **Collecte quotidienne** : une seule fois par jour, à 3 h (heure creuse), environ 70 pages de liste pour les 17 sites, avec exactement les mêmes délais et les mêmes règles ; le diagnostic n'est pas refait chaque nuit. |

**Contexte canadien.** Les sites de Gatineau relèvent de la Loi 25 (Québec) et ceux d'Ottawa de la
LPRPDE : aucune difficulté ici puisqu'aucun renseignement personnel n'est collecté. Pour le droit
d'auteur, le cours rappelle que les faits bruts (un prix, une date) ne sont pas protégés : nous
collectons des faits, ni photos ni textes rédigés. **Usage strictement pédagogique** : les données ne
seront ni publiées ni exploitées en dehors de ce travail.

> **Question 1 revérifiée le 2026-10-22** (portails ouvert.canada.ca et donneesquebec.ca) : toujours aucune API officielle ni donnée ouverte. La date est tenue à jour dans `config.VERIFICATION_API_MANUELLE`.

In [ ]:
print(config.VERIFICATION_API_MANUELLE)

### 1.2 Les 18 sites validés

Pour chaque site : sa plateforme (les sites d'une même plateforme ont le même HTML), son
Crawl-delay relevé dans robots.txt, et les règles relevées dans robots.txt et dans les conditions
d'utilisation (vérifiées le 9 octobre 2026). La colonne `collecter` vaut `False` pour un site noté
dans le journal mais jamais collecté.

In [ ]:
pd.DataFrame(config.SITES)[["nom", "ville", "plateforme", "delai_robots", "collecter", "regles"]]

### 1.3 Règle du groupe : zéro risque éthique

**En cas de doute sur la légalité ou l'éthique d'une collecte, on ne collecte pas.** Tout ce qui
est interdit, ou dont on ne peut pas être sûrs que c'est permis, est évité et noté dans le journal.

| Niveau | Sites ou pratiques | Raison |
|---|---|---|
| 🔴 **Interdit, on n'y touche pas** | Toyota Gatineau, Villa Toyota | Conditions SM360 : robots interdits |
| | Bel-Air Toyota, Car Canada, groupe Myers (16 concessionnaires) | Conditions : « spider, crawl, or scrape » interdits |
| | Ottawa Honda | Protection Cloudflare (403) : la contourner est interdit |
| | Pages détail et chemin `/json/` des sites D2C Media | Interdits par robots.txt |
| 🟠 **Évité par prudence** | Fichier `inventory-index.json` de Buckingham GM (cas E1b) | Fichier interne du chatbot : pas interdit mot à mot par robots.txt, mais on ne peut pas être sûrs que son usage par un robot est voulu |
| | Playwright pour charger le reste de l'inventaire D2C (cas E2) | Reviendrait à imiter un navigateur pour obtenir ce que le site ne sert pas aux robots |
| | Véhicules D2C absents du HTML (Buckingham 25, Hull Hyundai environ 9, Dupont 2) | Seule voie d'accès : JavaScript ou chemins interdits |
| | Téléphones, noms de vendeurs, descriptions rédigées, photos | Renseignements de contact et contenu protégé : on ne garde que des faits bruts |
| 🟢 **Permis** | HTML des listes d'inventaire des 17 sites accessibles | robots.txt le permet, aucune condition n'interdit les robots |
| | JSON-LD intégré dans ces pages (cas E1a) | Fait partie de la page déjà téléchargée, aucune requête de plus |
| | Pages suivantes par `?vip=` et `?pindex=` | Liens présents dans le HTML, non bloqués par robots.txt |

**Réserve.** Les sites Magnetis (3) et D2C Media (4) limitent l'usage du contenu à un « usage
personnel et non commercial ». Notre usage est pédagogique, non commercial et non publié, et ces
sites ont été validés par le professeur.

### 1.4 Sites écartés avant toute collecte

Ces sites ont été examinés puis écartés parce que leurs conditions d'utilisation interdisent la
collecte automatisée. Conformément aux consignes, ils sont notés mais ne comptent pas parmi nos
sites. Aucune requête automatisée ne leur est envoyée.

In [ ]:
pd.DataFrame(config.SITES_EXCLUS)

### 1.5 Liste des URL à traiter, sans doublons

Avant toute requête, `diagnostic.sites_sans_doublons()` retire les doublons : deux URL sont
identiques si elles ne diffèrent que par la casse du domaine ou par une barre oblique finale.
La même liste dédoublonnée sert au diagnostic (section 2) et à la collecte (section 3).

In [ ]:
SITES_A_TRAITER = diagnostic.sites_sans_doublons(config.SITES)
print(f"{len(config.SITES)} sites déclarés, {len(SITES_A_TRAITER)} URL uniques à traiter.")

## 2. Diagnostic (partie 2 du cours)

Pour chaque URL, `diagnostic.diagnostiquer_site()` applique l'arbre de décision du cours :

1. `ws.diagnostiquer(url)` lit robots.txt, envoie **une** requête et observe la page : statut
   HTTP, protection anti-robot, encodage, JavaScript, JSON intégré, tableaux, pagination, blocs
   répétés. Il en déduit le ou les cas (A à I).
2. Si le cas est **STOP** (robots.txt) ou **G** (blocage), on s'arrête : on ne contourne jamais.
3. Sinon, la page est téléchargée une fois avec `ws.obtenir()` (qui revérifie robots.txt, gère
   les erreurs 429 et 5xx, et s'arrête sur un 403) et enregistrée dans `pages_brutes/`.
4. `diagnostic.chercher_api()` cherche les signes d'**API** ou de données structurées (question 1
   de la partie 1, cas E) : JSON-LD, scripts `application/json`, adresses d'API dans le code, et
   permission robots.txt pour les chemins `/api/` et `/json/`.

Environ 3 minutes : 2 requêtes par site, avec le délai entre chaque.

In [ ]:
lignes_diagnostic = diagnostic.diagnostiquer_tous(SITES_A_TRAITER, session)
journal = diagnostic.construire_journal(lignes_diagnostic)

### 2.1 Résumé du diagnostic automatique

In [ ]:
journal.reindex(columns=["nom", "plateforme", "statut", "cas_diagnostic", "blocs_repetes",
                         "json_ld_vehicules", "robots_chemins_api"])

### 2.2 Du diagnostic automatique au cas confirmé

Le cours demande de confirmer chaque diagnostic en ouvrant la page dans le navigateur (F12).
Vérifications faites le 9 octobre 2026 dans Microsoft Edge :

- **Hyundai Gatineau** : 2 pages (`?vip=2`) ; Ctrl+U montre un JSON-LD `ItemList` de 20 véhicules.
- **Bank Street Mazda** : la liste se charge au défilement dans le navigateur, mais le HTML contient
  des liens `?pindex=` qui donnent les mêmes pages sans JavaScript.
- **Ottawa Honda** : la page s'ouvre lentement dans le navigateur (vérification Cloudflare
  silencieuse), alors que notre script reçoit un 403.
- **Buckingham GM et Hull Hyundai** : capture de l'onglet Réseau (fichier HAR). Aucune requête ne
  charge de véhicules au défilement. Chez Buckingham, la page charge le fichier interne du chatbot
  `inventory-index.json` (61 véhicules d'occasion et démos), **non utilisé par prudence** (section 1.3).

Les autres sites ont la même structure HTML que le site vérifié de leur plateforme ; leur cas
repose sur ce site de référence et sur le HTML enregistré.

**Les cas du cours appliqués à nos sites**

| Cas | Ce que ça veut dire | Sites concernés | Difficulté rencontrée | Ce qu'on fait |
|---|---|---|---|---|
| **A** HTML statique | Les véhicules sont dans le code HTML | Les 17 sites accessibles | Chaque plateforme a ses propres balises | Un lecteur de page par plateforme dans `extraction.py` |
| **B** Tableau HTML | Données dans un `<table>` | Aucun (fausse alerte sur 5 sites Dilawri et Hull Hyundai) | Le diagnostic a pris des tableaux de calcul de paiement pour des données | Cas écarté |
| **C** Pagination | Inventaire réparti sur plusieurs pages | Hyundai Gatineau, Gatineau Kia (`?vip=`), 10 sites Dilawri (`?pindex=`) | Défilement continu dans le navigateur ; liens de page non vus par le diagnostic | On suit les liens de page du HTML, avec délai et arrêt |
| **D** Liste vers pages détail | Une page par véhicule | Possible partout, interdit chez D2C Media | Une requête de plus par véhicule | Non utilisé : la liste suffit, et c'est plus poli |
| **E1a** JSON intégré | Données en JSON dans la page | Magnetis (`ItemList`), D2C Media (`Vehicle`) | Le JSON-LD ne contient pas le kilométrage | JSON-LD + HTML combinés, sans requête de plus |
| **E1b** API cachée | La page appelle un fichier JSON | Buckingham GM (`inventory-index.json`) | Fichier interne du chatbot, autorisation incertaine | Non utilisé par prudence |
| **E2** Navigateur automatisé | Playwright exécute le JavaScript | Aurait donné tout l'inventaire D2C | Revient à imiter un navigateur | Non utilisé |
| **F** Connexion | Compte requis | Aucun | | |
| **G** Blocage | Le site refuse l'accès | Ottawa Honda (403, Cloudflare) | Le site s'ouvre dans un vrai navigateur, pas pour notre script | Arrêt, aucun contournement |
| **H** Encodage | Accents mal décodés | 4 sites D2C Media (iso-8859-1) | Risque de « Ã© » au lieu de « é » | `corriger_encodage()` du cours (appelé par `obtenir()`) |
| **I** Grand volume | Scrapy | Aucun | Environ 60 pages et 900 véhicules | Une boucle requests avec délai suffit |

Les cas confirmés et la technique retenue pour chaque site sont dans `config.CAS_CONFIRMES` et
`config.TECHNIQUES` ; `diagnostic.construire_journal()` les a déjà reportés dans le journal.

In [ ]:
journal.reindex(columns=["nom", "cas_diagnostic", "cas_confirme", "technique"])

In [ ]:
chemin = diagnostic.enregistrer_journal(journal)
print("Journal enregistré :", chemin.relative_to(config.RACINE))

## 3. Collecte (parties 3 et 4 du cours)

### 3.1 Principe

Une **seule fonction**, `extraction.extraire_vehicules(site, session)`, est appelée **une fois par
URL** de la liste dédoublonnée (section 1.5). Pour chaque site, elle :

1. choisit le **lecteur de page** de sa plateforme (`lire_page_magnetis`, `lire_page_d2c`,
   `lire_page_dilawri`) : les sites d'une même plateforme ont le même HTML ;
2. télécharge la page avec `ws.obtenir()` : robots.txt revérifié, délai du site respecté,
   erreurs 429 et 5xx gérées, arrêt sur 403 ;
3. suit la **pagination présente dans le HTML** (`?vip=` ou `?pindex=`), en s'arrêtant dès qu'une
   page n'apporte aucun nouveau véhicule, que le total annoncé est atteint, ou après 20 pages ;
4. ne garde chaque véhicule **qu'une fois** (clé : NIV, sinon identifiant de l'annonce), car
   certaines pages Dilawri répètent des véhicules mis en vedette.

**Règle zéro risque appliquée au code** : pages de liste seulement (jamais les pages détail), seulement
le HTML servi par le site et le JSON-LD qu'il contient, ni API interne, ni navigateur automatisé, et
aucun renseignement personnel. Ottawa Honda (`collecter = False`) n'est jamais contacté.

**Colonnes produites** (une ligne par véhicule)

| Groupe | Colonnes |
|---|---|
| Site | `site_id`, `concessionnaire`, `ville`, `province`, `plateforme`, `date_collecte` |
| Annonce | `url_page`, `url_fiche`, `id_annonce`, `no_stock`, `niv` |
| Véhicule | `annee`, `marque`, `modele`, `version`, `etat` |
| Prix | `prix_affiche`, `rabais`, `frais`, `prix_final` |
| Caractéristiques | `km`, `transmission`, `traction`, `carburant`, `moteur`, `couleur_ext`, `categorie` |

**Prix selon la plateforme** (à harmoniser au nettoyage, section 4) :

| Plateforme | `prix_affiche` | `rabais` | `frais` | `prix_final` |
|---|---|---|---|---|
| Magnetis | « Price » | « Rebate » | absent | « Your price » |
| D2C Media | ancien prix barré (sinon prix actuel) | différence | absent | prix actuel |
| Dilawri | « Price » du détail de prix | réductions | Admin Fee, OMVIC Fee | « Dilawri selling price » |

Durée : environ 5 à 10 minutes (environ 60 pages, 2 à 5 s entre chaque).

In [ ]:
vehicules, rapports = extraction.collecter_tous(SITES_A_TRAITER, session)

### 3.2 Résultat par site

In [ ]:
resume_collecte = pd.DataFrame(rapports)
resume_collecte["remarques"] = resume_collecte["remarques"].apply(" ".join)
resume_collecte.reindex(columns=["id", "nb_lignes", "pages_collectees", "fichier_csv", "remarques"])

In [ ]:
print(f"{len(vehicules)} véhicules collectés sur {vehicules['site_id'].nunique()} sites")
vehicules.groupby(["province", "plateforme"]).size().rename("véhicules").to_frame()

In [ ]:
vehicules.head(10)

### 3.3 Enregistrement

- un CSV par site dans `resultats/brut/` (déjà écrit par `collecter_tous()`) ;
- un fichier de tous les véhicules de la collecte : `resultats/vehicules_brut_AAAA-MM-JJ.csv` ;
- le journal de tests, complété avec `nb_lignes`, les pages collectées, le fichier et les remarques.

Encodage `utf-8-sig` partout, pour une lecture correcte dans Excel.

In [ ]:
jour = datetime.now().strftime("%Y-%m-%d")
fichier_vehicules = config.DOSSIER_RESULTATS / f"vehicules_brut_{jour}.csv"
vehicules.to_csv(fichier_vehicules, index=False, encoding="utf-8-sig")
print("Véhicules :", fichier_vehicules.relative_to(config.RACINE))

journal = extraction.ajouter_collecte_au_journal(journal, rapports)
chemin = diagnostic.enregistrer_journal(journal)
print("Journal   :", chemin.relative_to(config.RACINE))
journal.reindex(columns=["nom", "nb_lignes", "pages_collectees", "problemes_remarques"])

## 4. Un paragraphe par URL (partie 5 du cours)

Pour chaque URL de la liste dédoublonnée : le cas détecté (diagnostic automatique, puis confirmé dans le
navigateur avec F12), la technique choisie et les difficultés rencontrées. Chiffres de la collecte du 2026-10-22.

**Nettoyage et fusion** : voir le notebook `02_pretraitement.ipynb`. **Documentation technique complète**
(fiches par plateforme, cadre légal, rôle de chaque fichier) : document « Documentation technique du projet de web scraping ».

### Magnetis (Gatineau)

**Hyundai Gatineau** (`https://hyundaigatineau.ca/en/pre-owned`). Cas A + C + E1a. Les véhicules sont des blocs
`div.vehicle-card-wrapper` dont les attributs `data-vehicle-*` donnent directement le NIV, l'année, la marque et le
modèle ; la page contient aussi un JSON-LD `ItemList` de 20 véhicules. Technique : requests + BeautifulSoup sur les
attributs, JSON-LD en complément pour le prix et le lien, pagination par le lien `?vip=2` présent dans le HTML.
34 véhicules sur 2 pages. Difficulté : ni catégorie ni couleur publiées (catégorie déduite au prétraitement).

**Gatineau Acura** (`https://gatineauacura.com/en/pre-owned`). Cas A + E1a. Même plateforme et même lecteur que
Hyundai Gatineau ; tout l'inventaire tient sur une seule page (29 fiches, ItemList de 29 véhicules), donc aucune
pagination. 29 véhicules. Aucune difficulté particulière.

**Gatineau Kia** (`https://www.gatineaukia.ca/en/pre-owned`). Cas A + C + E1a. Même plateforme ; pagination `?vip=`
suivie jusqu'à la dernière page. 47 véhicules sur 3 pages. Difficulté : 2 véhicules sans prix affiché, gardés et
signalés (`prix_non_affiche`).

### D2C Media (Gatineau)

**Dupont & Dupont Honda** (`https://www.dupontduponthonda.com/occasion/recherche.html`). Cas A + H. Fiches
`li.carBoxWrapper`, état dans `input[data-condition]`, JSON-LD `Vehicle` par fiche ; page en iso-8859-1, corrigée
avec les outils du cours. robots.txt impose 5 s entre les requêtes et interdit les pages détail et `/json/`.
Technique : une seule page de liste, telle que servie. 36 véhicules sur 38 annoncés : les 2 autres ne sont
accessibles que par JavaScript ou par des chemins interdits, donc non collectés.

**Hull Hyundai** (`https://www.hullhyundai.com/occasion/recherche.html`). Cas A + H. Même plateforme et mêmes règles.
L'enregistrement HAR (F12, onglet Réseau) montre qu'aucune requête n'est faite au défilement : seules les fiches du
HTML sont accessibles proprement. 36 véhicules. Difficulté : inventaire partiel, accepté par la règle zéro risque.

**Hull Nissan** (`https://www.hullnissangatineau.ca/occasion/recherche.html`). Cas A + H. Même plateforme ; les
12 véhicules annoncés sont tous dans le HTML : inventaire complet. 12 véhicules. Seule difficulté : l'encodage
iso-8859-1, corrigé.

**Buckingham Chevrolet Buick GMC** (`https://www.buckinghamgm.com/occasion/recherche.html`). Cas A + H. Même
plateforme. La page charge un fichier interne `inventory-index.json` (61 occasions et démos) destiné au robot de
clavardage : il n'est pas utilisé, car son usage par un robot n'est pas clairement autorisé. 36 véhicules, ceux du HTML.

### EDealer (Ottawa)

**Ottawa Honda** (`https://www.ottawahonda.com/inventory/used/`). Cas G. Le serveur répond 403 avec un défi
Cloudflare « Just a moment » ; robots.txt autorise pourtant `/inventory/`. Technique : aucune. Arrêt immédiat,
aucune tentative de contournement (`collecter = False`) ; le site reste dans le journal pour documenter le cas G.
0 véhicule.

### Dilawri (Ottawa, Groupe Dilawri)

Les 10 sites partagent la même plateforme : fiches `div.vsearchresult` avec attributs `data-chrome-*` (stock,
année, marque, modèle, prix de vente), fiche technique en liste `dt`/`dd`, ventilation du prix dans l'attribut
`data-content` d'un bouton. Le navigateur affiche un défilement continu, mais le HTML contient des liens
`?pindex=` : technique requests + BeautifulSoup avec pagination par ces liens et 2 s entre les pages (cas A + C).
robots.txt ne bloque que `/oops` et `/signalr/` ; aucune condition d'utilisation trouvée. Difficultés communes :
la page 1 utilise un autre tri que les liens de pagination (environ 12 véhicules manqués par site le 2026-10-09,
corrigé le 2026-10-22 en relisant la page 1 dans le même tri), des bannières publicitaires mêlées aux fiches
(ignorées), et le NIV précédé de « VIN: » (nettoyé).

**Bank Street Mazda** (`https://www.bankstreetmazda.com/ottawa-Mazda-dealer/used-cars/used-vehicle-inventory?type=U`).
Cas A + C. 80 véhicules sur 7 pages. Difficulté propre : 5 véhicules à 0 $ (« appelez-nous »), convertis en prix manquant.

**Bank Street Hyundai** (`https://www.bankstreethyundai.com/ottawa-gloucester-hyundai-dealer/used-cars/used-vehicle-inventory?type=U`).
Cas A + C. 55 véhicules sur 5 pages. Difficulté propre : 821 $ de frais inclus dans le prix final, isolés dans la
colonne `frais` (prix + frais - rabais = prix final, vérifié).

**Bank Street Kia** (`https://www.bankstreetkia.com/ottawa-Kia-dealer/used-cars/used-vehicle-inventory?type=U`).
Cas A + C. 38 véhicules sur 4 pages. Les tableaux détectés par le diagnostic sont des calculateurs de paiement :
cas B écarté.

**Bank Street Mitsubishi** (`https://www.bankstreetmitsubishi.com/ottawa-Mitsubishi-dealer/used-cars/used-vehicle-inventory?type=U`).
Cas A + C. 49 véhicules sur 5 pages. Cas B écarté pour la même raison.

**Barrhaven Mazda** (`https://www.barrhavenmazda.com/ottawa/used/inventory?type=U`). Cas A + C. 27 véhicules
sur 3 pages. Aucune difficulté propre.

**Barrhaven Chrysler Jeep Dodge Ram** (`https://www.barrhavenjeepdodgeram.com/used-vehicles/used-vehicle-inventory?type=U`).
Cas A + C. 93 véhicules sur 8 pages. Difficulté propre : 22 $ de frais inclus dans le prix final, isolés dans `frais`.

**Dilawri Chrysler Jeep Dodge Ram** (`https://www.dilawrichrysler.com/chrysler-jeep-dodge-ram-dealer-ottawa/used-cars/used-vehicle-inventory?type=U`).
Cas A + C. 101 véhicules sur 9 pages, le plus gros inventaire. 1 véhicule à 0 $.

**Orleans Mitsubishi** (`https://www.orleansmitsubishi.com/mitsubishi/used-inventory.html?type=U`). Cas A + C.
48 véhicules sur 4 pages. robots.txt bloque aussi le blogue, sans effet sur l'inventaire ; cas B écarté.

**Hyundai On Hunt Club** (`https://www.hyundaionhuntclub.com/hyundai/used~vehicles/inventory.html?type=U`).
Cas A + C. 28 véhicules sur 3 pages. 2 véhicules à 0 $.

**Kia On Hunt Club** (`https://www.kiaonhuntclub.com/kia/used-inventory.html?type=U`). Cas A + C. 52 véhicules
sur 5 pages. JSON-LD présent mais illisible et sans véhicules : non utilisé ; robots.txt bloque le blogue ;
2 véhicules à 0 $.

## 5. Vérification finale

**Run All** dans un noyau neuf, sans erreur, pour les deux notebooks.